### RLPINNER ENV

In [88]:

import numpy as np
from math import sqrt
import warnings
import copy
import pandas as pd
from pandas.core.common import SettingWithCopyWarning
from utils import *
from pre_processing import *
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)

In [110]:
data1 = pd.read_csv("data/symbol_pins.csv")
symbols=get_neighbors_partnumbers(data1)
#symbols = get_pin_int_out(data1, symbols)

data2= pd.read_csv("data/inline_pins.csv")

In [105]:
data2=data2[data2['Connector Name']=='ILC8A']

In [106]:
data2.to_csv("data/new_connector.csv", index=False)

In [92]:
data2[['Connector PartNumber']].isnull().sum().sum()

0

In [111]:
data, d, max_categories = preprocess_data(data2, symbols)

In [101]:
data

,Connector Name,Pin Name,Pin Name nunique,Pin Name count,Connector PartNumber,Pin PreferredSignal,Signal Name,Wire WireColor,Wire WireCSA,MulticoreInnerToOutter1,...,Pin Height,distance_to_center,neighbors,neighbors_length,internal_pin,mean_neighbors,centroide_distance,number_internal_pin,min_distance,max_distance
0,ILC8A,1,1,1,C-81440-D,na,2N-AUDIO-COMMON,G/W,0.35,na,...,0.0,7595.535531,"[2, 11, 12]",3,False,6861.292983,6841.118037,21,3396.826754,14863.578035
1,ILC8A,2,1,1,C-81440-D,na,2N-AUDIO-LEFT,G/O,0.35,na,...,0.0,8075.420484,"[1, 3, 12, 13]",4,True,7484.513495,7433.803872,21,3396.826754,14863.578035
2,ILC8A,3,1,1,C-81440-D,na,2N-AUDIO-RIGHT,G/B,0.35,na,...,0.0,8682.561834,"[2, 4, 13, 14, 12]",5,False,7827.789327,7780.147042,21,3396.826754,14863.578035
3,ILC8A,4,1,1,C-81440-D,na,2N-AUX-CTL-1,N/W,0.35,na,...,0.0,9392.313453,"[5, 3, 14, 15]",4,True,8938.194627,8892.833969,21,3396.826754,14863.578035
4,ILC8A,5,1,1,C-81440-D,na,2N-AUX-CTL-2,N/O,0.35,na,...,0.0,10183.242706,"[4, 6, 15, 16, 14]",5,True,9391.869549,9359.679995,21,3396.826754,14863.578035
5,ILC8A,6,1,1,C-81440-D,na,2N-AUX_AUDIO-1,B/W,0.35,na,...,0.0,11037.912846,"[7, 5, 16, 17, 15]",5,True,10282.234506,10252.87191,21,3396.826754,14863.578035
6,ILC8A,7,1,1,C-81440-D,na,2N-AUX_AUDIO-2,B/O,0.35,na,...,0.0,11942.646943,"[8, 6, 17, 18]",4,True,11592.784851,11556.442359,21,3396.826754,14863.578035
7,ILC8A,8,1,1,C-81440-D,na,2N-AUX_AUDIO-3,B/R,0.35,na,...,0.0,12886.904981,"[7, 9, 18, 19, 17]",5,True,12192.848659,12164.80242,21,3396.826754,14863.578035
8,ILC8A,9,1,1,C-81440-D,na,2N-AUX_AUDIO-4,B/U,0.35,na,...,0.0,13862.612741,"[10, 8, 20, 19]",4,True,13576.116883,13547.654557,21,3396.826754,14863.578035
9,ILC8A,10,1,1,C-81440-D,na,2N-GND-1,B,0.35,na,...,0.0,14863.578035,"[9, 20]",2,True,13851.96761,13825.333269,21,3396.826754,14863.578035


In [86]:
list(data['Connector Name'].unique())[0]

'TO_183_F/162_13P'

In [43]:
data.to_csv("data/new_connector.csv")

In [107]:
def get_models_sizes(connectors_dicc):
    """
    create a vector with all posible sizes of connectors and a dicctionary with key num_pins and a vector
    with the names of connectors that have this pins

    :param connectors_dicc:
    :return:
    """
    d_models = []
    model_connectors = {}

    for connector in connectors_dicc.keys():
        num_pins = connectors_dicc[connector]['1']['Num_Pins']

        if not (num_pins in d_models):
            d_models.append(num_pins)

            model_connectors[num_pins] = []
        if not (connector in model_connectors[num_pins]):
            model_connectors[num_pins].append(connector)
    return d_models, model_connectors

In [ ]:
def get_models_sizes(connectors_dicc):
    """
    create a vector with all posible sizes of connectors and a dicctionary with key num_pins and a vector
    with the names of connectors that have this pins

    :param connectors_dicc:
    :return:
    """
    d_models = []
    model_connectors = {}

    for connector in connectors_dicc.keys():
        num_pins = connectors_dicc[connector]['1']['Num_Pins']

        if not (num_pins in d_models):
            d_models.append(num_pins)

            model_connectors[num_pins] = []
        if not (connector in model_connectors[num_pins]):
            model_connectors[num_pins].append(connector)
    return d_models, model_connectors

In [112]:
connectors_dicc=create_connector_dicc(data)

In [113]:
get_models_sizes(connectors_dicc)

([2, 26, 20, 40],
 {2: ['ILC10A', 'ILC10B', 'ILC5A', 'ILC5B', 'ILC9A', 'ILC9B'],
  26: ['ILC1A',
   'ILC1B',
   'ILC2A',
   'ILC2B',
   'ILC3A',
   'ILC3B',
   'ILC4A',
   'ILC4B',
   'ILC6A',
   'ILC6B'],
  20: ['ILC7A', 'ILC7B'],
  40: ['ILC8A', 'ILC8B']})

In [98]:
connector='ILC8A'

In [102]:
dicc_signals = eval(connectors_dicc[connector]['1']['dicc_signals'])

In [103]:
dicc_signals

{0: '10-AJARLT-1',
 1: '10-AJARLTRR-1',
 2: '10-AJARRT-1',
 3: '10-AJARRTRR-1',
 4: '2N-AUDIO-COMMON',
 5: '2N-AUDIO-LEFT',
 6: '2N-AUDIO-RIGHT',
 7: '2N-AUX-CTL-1',
 8: '2N-AUX-CTL-2',
 9: '2N-AUX_AUDIO-1',
 10: '2N-AUX_AUDIO-2',
 11: '2N-AUX_AUDIO-3',
 12: '2N-AUX_AUDIO-4',
 13: '2N-GND-1',
 14: '2N-POWER-3',
 15: '2N-SBWFR-1',
 16: '2N-SBWFR-2',
 17: '2N-SPKR-LR-1',
 18: '2N-SPKR-LR-2',
 19: '2N-SPKR-LT-1',
 20: '2N-SPKR-LT-2',
 21: '2N-SPKR-RR-1',
 22: '2N-SPKR-RR-2',
 23: '2N-SPKR-RT-1',
 24: '2N-SPKR-RT-2',
 25: '3N-PARKRR-1',
 26: '3N-STOP-1',
 27: '3N-STOP-2',
 28: '3N-TURNLTRR-1',
 29: '3N-TURNRTRR-1',
 30: '4N-FUEL-LID',
 31: '4N-TRUNK',
 32: 'na'}

In [78]:
multicore=[]
for key in connectors_dicc[connector].keys():
    if key != 'PartNumber':
        multicore.append(connectors_dicc[connector][str(key)]['is_multicore'])

In [79]:
sum(multicore)

4

In [76]:
connectors_dicc[connector].keys()

dict_keys(['1', 'PartNumber', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13'])

In [47]:
num_signals = len(dicc_signals.keys())
num_signals

9

In [49]:
Num_Pins = connectors_dicc[connector]['1']['Num_Pins']
Num_Pins

13

In [50]:
number_possible_variations=num_signals**Num_Pins
number_possible_variations

2541865828329

In [55]:
len(str(number_possible_variations))-5

8

In [56]:
int(number_possible_variations/ 10 **8)

25418

In [10]:
Num_Pins = connectors_dicc['TO_121_TNGA_F/219_13P']['1']['Num_Pins']

In [11]:
dicc_signals= eval(connectors_dicc['TO_121_TNGA_F/219_13P']['1']['dicc_signals'])

In [12]:
num_signals= len(dicc_signals.keys())

In [13]:
dicc_signals_copy=copy.deepcopy(dicc_signals)

In [14]:
def getKey(dct,value):
     return [key for key in dct if (dct[key] == value)]

In [15]:
getKey(dicc_signals_copy, 'na')[0]

8

In [16]:
if "na" in list(dicc_signals.values()):
    key=getKey(dicc_signals_copy, 'na')[0]
    dicc_signals_copy.pop(key, None)
    

In [17]:
signals=list(dicc_signals_copy.values())
signals

['EQ_FUEL_PRESS_SSR_HI-TNGA_E2',
 'EQ_FUEL_PRESS_SSR_HI-TNGA_PR',
 'EQ_FUEL_PRESS_SSR_HI-TNGA_VC',
 'EQ_NE_SSR-TNGA_NE-',
 'EQ_NE_SSR-TNGA_NEP',
 'EQ_NE_SSR-TNGA_VCNE',
 'SLD_EFI_ECU-TNGA_KNOCK_SSR_101',
 'SLD_EFI_ECU-TNGA_KNOCK_SSR_102']

In [18]:
conn=np.zeros((Num_Pins), dtype="int")
conn

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [19]:
cols = [str(i) for i in range(1, Num_Pins + 1)]
data_test = pd.DataFrame(data=[], columns=cols)

for x in range(0, 5000):
    signals=list(dicc_signals_copy.values())
    conn=np.zeros((Num_Pins), dtype="int")
    pin_used=[]
    breaker=False
    data = {}
    while breaker ==False:
        pin=random.randint(0, Num_Pins-1 )
        if not(pin in pin_used):
            pin_used.append(pin)
            if len(signals) > 0:
                pos = random.randint(0, len(signals)-1 )
                conn[pin]=getKey(dicc_signals, signals[pos])[0]
                data[str(pin + 1)] =getKey(dicc_signals, signals[pos])[0]
                #print(signals[pos])
                signals.remove(signals[pos])
            else:
                data[str(pin + 1)] =getKey(dicc_signals, 'na')[0]
                #print("na")
        if len(pin_used)==Num_Pins:
            data_test = data_test.append(data, ignore_index=True)
            breaker = True


In [20]:
data_test

,1,2,3,4,5,6,7,8,9,10,11,12,13
0,7,8,8,0,6,5,4,1,3,2,8,8,8
1,6,8,8,7,8,1,0,8,4,3,8,5,2
2,6,8,5,8,8,4,2,8,1,7,3,8,0
3,6,0,3,1,8,8,2,4,8,8,7,8,5
4,0,7,3,4,5,6,8,8,8,2,1,8,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,3,0,7,6,2,4,8,8,5,8,1,8,8
4996,1,8,0,5,6,8,7,8,8,3,4,2,8
4997,2,0,6,4,8,3,5,7,8,1,8,8,8
4998,1,8,6,0,8,8,4,3,7,8,2,8,5


In [21]:
for col in data_test.columns:
    data_test["cat_" + str(col)] = data_test.apply(lambda row: return_cat(row, col, dicc_signals), axis=1)
    data_test[col] = data_test.apply(lambda row: category_signal_test(row, col), axis=1)

In [22]:
dicc_signals

{0: 'EQ_FUEL_PRESS_SSR_HI-TNGA_E2',
 1: 'EQ_FUEL_PRESS_SSR_HI-TNGA_PR',
 2: 'EQ_FUEL_PRESS_SSR_HI-TNGA_VC',
 3: 'EQ_NE_SSR-TNGA_NE-',
 4: 'EQ_NE_SSR-TNGA_NEP',
 5: 'EQ_NE_SSR-TNGA_VCNE',
 6: 'SLD_EFI_ECU-TNGA_KNOCK_SSR_101',
 7: 'SLD_EFI_ECU-TNGA_KNOCK_SSR_102',
 8: 'na'}

In [23]:
data_test.head()

,1,2,3,4,5,6,7,8,9,10,...,cat_4,cat_5,cat_6,cat_7,cat_8,cat_9,cat_10,cat_11,cat_12,cat_13
0,7,8,8,7,7,7,7,7,7,7,...,EQ_FUEL_PRESS_SSR_HI-TNGA_E2,SLD_EFI_ECU-TNGA_KNOCK_SSR_101,EQ_NE_SSR-TNGA_VCNE,EQ_NE_SSR-TNGA_NEP,EQ_FUEL_PRESS_SSR_HI-TNGA_PR,EQ_NE_SSR-TNGA_NE-,EQ_FUEL_PRESS_SSR_HI-TNGA_VC,na,na,na
1,7,8,8,7,8,7,7,8,7,7,...,SLD_EFI_ECU-TNGA_KNOCK_SSR_102,na,EQ_FUEL_PRESS_SSR_HI-TNGA_PR,EQ_FUEL_PRESS_SSR_HI-TNGA_E2,na,EQ_NE_SSR-TNGA_NEP,EQ_NE_SSR-TNGA_NE-,na,EQ_NE_SSR-TNGA_VCNE,EQ_FUEL_PRESS_SSR_HI-TNGA_VC
2,7,8,7,8,8,7,7,8,7,7,...,na,na,EQ_NE_SSR-TNGA_NEP,EQ_FUEL_PRESS_SSR_HI-TNGA_VC,na,EQ_FUEL_PRESS_SSR_HI-TNGA_PR,SLD_EFI_ECU-TNGA_KNOCK_SSR_102,EQ_NE_SSR-TNGA_NE-,na,EQ_FUEL_PRESS_SSR_HI-TNGA_E2
3,7,7,7,7,8,8,7,7,8,8,...,EQ_FUEL_PRESS_SSR_HI-TNGA_PR,na,na,EQ_FUEL_PRESS_SSR_HI-TNGA_VC,EQ_NE_SSR-TNGA_NEP,na,na,SLD_EFI_ECU-TNGA_KNOCK_SSR_102,na,EQ_NE_SSR-TNGA_VCNE
4,7,7,7,7,7,7,8,8,8,7,...,EQ_NE_SSR-TNGA_NEP,EQ_NE_SSR-TNGA_VCNE,SLD_EFI_ECU-TNGA_KNOCK_SSR_101,na,na,na,EQ_FUEL_PRESS_SSR_HI-TNGA_VC,EQ_FUEL_PRESS_SSR_HI-TNGA_PR,na,na


In [24]:
data_test.to_csv("data_test.csv")

In [ ]:
random.randint(0, num_signals - 1)

In [26]:
dt=data_test.sort_values("1", ascending=False )

In [27]:
dt[:1]

,1,2,3,4,5,6,7,8,9,10,...,cat_4,cat_5,cat_6,cat_7,cat_8,cat_9,cat_10,cat_11,cat_12,cat_13
4999,8,7,7,7,7,8,7,8,7,8,...,EQ_FUEL_PRESS_SSR_HI-TNGA_PR,EQ_FUEL_PRESS_SSR_HI-TNGA_VC,na,EQ_FUEL_PRESS_SSR_HI-TNGA_E2,na,EQ_NE_SSR-TNGA_NEP,na,SLD_EFI_ECU-TNGA_KNOCK_SSR_101,na,EQ_NE_SSR-TNGA_NE-
